# Lab 5 — Workflow Design with LLMs (Prompt Templates & Output Parsers)

**Subject:** Generative AI (GenAI)
**Repository:** `generative_ai_<student-name-or-roll-no>`

---
## Aim
To build multi-step LLM workflows in LangChain where every step uses a prompt template and an output parser, so that the parsed output of one step can be passed cleanly into the prompt of the next step.

## Theory
- **PromptTemplate** is a prompt with blanks like `{review_text}` that get filled in at run time. It is used for simple, single-text prompts.
- **ChatPromptTemplate** builds a prompt from messages (system, human). The *system* message lets us give the model a persona, such as "support ticket writer".
- **StrOutputParser** just returns the model's reply as a plain string. Good for free-form text.
- **PydanticOutputParser** forces the reply into a Pydantic class (a fixed schema with types). It also gives us *format instructions* that we add to the prompt, so the model replies in the correct JSON format. This gives type-safe data that is easy to pass to the next step.

**Rule followed in this lab**

| Type of step | Template | Parser |
|---|---|---|
| Extract structured data from raw text | `PromptTemplate` | `PydanticOutputParser` |
| Generate free-form text | `ChatPromptTemplate` (with persona) | `StrOutputParser` |
| Evaluate / flag issues / final structured output | `ChatPromptTemplate` (with persona) | `PydanticOutputParser` |

Each step is written as a small chain: `prompt | llm | parser`.

## Required Libraries

In [1]:
# Run this cell once (Colab / Jupyter)
!pip install -q langchain langchain-core langchain-google-genai pydantic requests

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.6/58.6 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.9/81.9 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 43.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 267.9/267.9 kB 16.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.49.0, but you have google-auth 2.62.0 which is incompatible.


In [2]:
import os
import json
import requests
from datetime import date
from getpass import getpass
from typing import List

from pydantic import BaseModel, Field
from langchain_core.prompts import PromptTemplate, ChatPromptTemplate
from langchain_core.output_parsers import PydanticOutputParser, StrOutputParser
from langchain_google_genai import ChatGoogleGenerativeAI

### LLM setup
I am using Google Gemini through LangChain. It needs an API key (free from Google AI Studio). If you use some other provider, only the `llm = ...` line needs to change; the rest of the notebook stays the same.

In [3]:
if "GOOGLE_API_KEY" not in os.environ:
    os.environ["GOOGLE_API_KEY"] = getpass("Enter your Google API key: ")

# temperature=0 so that the extraction steps give stable results
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0)

Enter your Google API key: ··········


---
## Exercise 1 — Customer Review to Support Ticket Pipeline

**Question:** Build a 2-step pipeline. Step 1 extracts the core complaint, product/feature mentioned and customer sentiment from a raw customer review as structured data. Step 2 generates a support ticket summary for the internal team using only the structured data from Step 1, not the raw review.

**Input:** `review_text`
**Output:** structured complaint data (Step 1) and support ticket summary (Step 2)

**My approach**
- Step 1: `PromptTemplate` + `PydanticOutputParser` (`complaint`, `product_or_feature`, `sentiment`).
- Step 2: `ChatPromptTemplate` with the persona "support ticket writer" + `StrOutputParser`.
- Only the Step 1 JSON is sent to Step 2, the raw review is not.

In [4]:
# ----- Step 1: extract complaint data -----
class ComplaintData(BaseModel):
    complaint: str = Field(description="The main complaint of the customer in one or two sentences")
    product_or_feature: str = Field(description="The product or feature the customer is talking about")
    sentiment: str = Field(description="Customer sentiment: positive, neutral or negative")

ex1_parser = PydanticOutputParser(pydantic_object=ComplaintData)

ex1_step1_prompt = PromptTemplate(
    template=(
        "Read the customer review below and extract the complaint details.\n"
        "{format_instructions}\n\n"
        "Review:\n{review_text}"
    ),
    input_variables=["review_text"],
    partial_variables={"format_instructions": ex1_parser.get_format_instructions()},
)

ex1_step1_chain = ex1_step1_prompt | llm | ex1_parser

# ----- Step 2: write support ticket summary (uses only Step 1 data) -----
ex1_step2_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a support ticket writer. You write short, clear ticket summaries for the internal support team."),
    ("human", "Using only the structured complaint data below, write a support ticket summary "
              "(a title, a short description and a suggested priority).\n\n{complaint_data}"),
])

ex1_step2_chain = ex1_step2_prompt | llm | StrOutputParser()


def run_exercise_1(review_text):
    complaint_data = ex1_step1_chain.invoke({"review_text": review_text})
    ticket = ex1_step2_chain.invoke({"complaint_data": complaint_data.model_dump_json(indent=2)})
    return complaint_data, ticket

In [5]:
review_text = """I bought the Zenith X200 wireless earbuds two weeks back and honestly I'm really disappointed.
The left earbud keeps disconnecting from my phone in the middle of calls, and the battery
hardly lasts 2 hours even though the box says 8 hours. I wrote to support once and got no reply.
Not worth the money."""

complaint_data, ticket = run_exercise_1(review_text)

print("STEP 1 - Structured complaint data")
print(complaint_data.model_dump_json(indent=2))
print("\nSTEP 2 - Support ticket summary")
print(ticket)

GoogleInvalidRequestError: Error calling model 'gemini-2.5-flash' (INVALID_ARGUMENT): 400 INVALID_ARGUMENT. {'error': {'code': 400, 'message': 'API key not valid. Please pass a valid API key.', 'status': 'INVALID_ARGUMENT', 'details': [{'@type': 'type.googleapis.com/google.rpc.ErrorInfo', 'reason': 'API_KEY_INVALID', 'domain': 'googleapis.com', 'metadata': {'service': 'generativelanguage.googleapis.com'}}, {'@type': 'type.googleapis.com/google.rpc.LocalizedMessage', 'locale': 'en-US', 'message': 'API key not valid. Please pass a valid API key.'}]}}

---
## Exercise 2 — Research Paper Abstract to Layperson Summary Pipeline

**Question:** Build a 2-step pipeline. Step 1 extracts the research question, method and key finding from a paper abstract as structured data. Step 2 generates a plain-language summary for a non-expert audience, using only the structured version from Step 1, not the original abstract.

**Input:** `abstract_text`
**Output:** structured extraction (Step 1) and layperson summary (Step 2)

**My approach**
- Step 1: `PromptTemplate` + `PydanticOutputParser` (`research_question`, `method`, `key_finding`).
- Step 2: `ChatPromptTemplate` with the persona "science communicator explaining to a general audience" + `StrOutputParser`.

In [ ]:
# ----- Step 1: extract from abstract -----
class AbstractData(BaseModel):
    research_question: str = Field(description="What question the researchers wanted to answer")
    method: str = Field(description="How the researchers did the study")
    key_finding: str = Field(description="The main result of the study")

ex2_parser = PydanticOutputParser(pydantic_object=AbstractData)

ex2_step1_prompt = PromptTemplate(
    template=(
        "Read the research paper abstract below and extract the research question, "
        "the method used and the key finding.\n"
        "{format_instructions}\n\n"
        "Abstract:\n{abstract_text}"
    ),
    input_variables=["abstract_text"],
    partial_variables={"format_instructions": ex2_parser.get_format_instructions()},
)

ex2_step1_chain = ex2_step1_prompt | llm | ex2_parser

# ----- Step 2: layperson summary (uses only Step 1 data) -----
ex2_step2_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a science communicator explaining research to a general audience. "
               "Use simple words, avoid jargon and keep it short and interesting."),
    ("human", "Write a plain-language summary (about 4-5 sentences) using only this structured information:\n\n{structured_data}"),
])

ex2_step2_chain = ex2_step2_prompt | llm | StrOutputParser()


def run_exercise_2(abstract_text):
    structured = ex2_step1_chain.invoke({"abstract_text": abstract_text})
    summary = ex2_step2_chain.invoke({"structured_data": structured.model_dump_json(indent=2)})
    return structured, summary

In [ ]:
abstract_text = """Sleep deprivation is widely believed to reduce academic performance, but the effect on
exam results of university students is not well measured. We conducted a randomized study with
240 undergraduate students who were assigned to either a fixed 8-hour sleep schedule or their
normal sleep routine for four weeks, and compared their scores in three standardized tests.
Students on the fixed sleep schedule scored on average 11% higher than the control group,
and the improvement was largest in subjects that need memory and problem solving."""

structured, summary = run_exercise_2(abstract_text)

print("STEP 1 - Structured extraction")
print(structured.model_dump_json(indent=2))
print("\nSTEP 2 - Layperson summary")
print(summary)

---
## Exercise 3 — Interview Transcript to Candidate Scorecard Pipeline

**Question:** Build a 3-step pipeline. Step 1 extracts the candidate's answers to each interview question from a raw transcript. Step 2 evaluates each answer against a given set of skills, flagging any skill that was not clearly demonstrated. Step 3 formats the evaluation into a structured hiring scorecard with an overall recommendation.

**Input:** `transcript_text`, `skills_to_assess`
**Output:** extracted answers (Step 1), flagged skill evaluation (Step 2), final scorecard (Step 3)

**My approach**
- Step 1: `PromptTemplate` + `PydanticOutputParser` (list of `question` + `answer`).
- Step 2: `ChatPromptTemplate` ("technical interviewer evaluating skill evidence") + `PydanticOutputParser` (`skill`, `demonstrated`, `evidence_or_gap` for each skill).
- Step 3: `ChatPromptTemplate` ("hiring panel coordinator") + `PydanticOutputParser` (scorecard rows + `overall_recommendation`).

In [ ]:
# ----- Step 1: extract question-answer pairs -----
class QAPair(BaseModel):
    question: str = Field(description="Question asked by the interviewer")
    answer: str = Field(description="Answer given by the candidate, summarised faithfully")

class ExtractedAnswers(BaseModel):
    qa_pairs: List[QAPair] = Field(description="All question and answer pairs from the transcript")

ex3_parser1 = PydanticOutputParser(pydantic_object=ExtractedAnswers)

ex3_step1_prompt = PromptTemplate(
    template=(
        "Below is a raw interview transcript. Extract every question the interviewer asked "
        "and the answer the candidate gave.\n"
        "{format_instructions}\n\n"
        "Transcript:\n{transcript_text}"
    ),
    input_variables=["transcript_text"],
    partial_variables={"format_instructions": ex3_parser1.get_format_instructions()},
)
ex3_step1_chain = ex3_step1_prompt | llm | ex3_parser1

# ----- Step 2: evaluate each skill, flag the ones not demonstrated -----
class SkillEvaluation(BaseModel):
    skill: str = Field(description="Name of the skill being assessed")
    demonstrated: bool = Field(description="True only if the candidate clearly demonstrated the skill")
    evidence_or_gap: str = Field(description="Evidence from the answers if demonstrated, otherwise what is missing")

class EvaluationResult(BaseModel):
    evaluations: List[SkillEvaluation] = Field(description="One entry for each skill to assess")

ex3_parser2 = PydanticOutputParser(pydantic_object=EvaluationResult)

ex3_step2_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a technical interviewer evaluating skill evidence. "
               "Mark a skill as demonstrated only if the answers clearly show it. "
               "If the evidence is weak, vague or missing, set demonstrated to false and explain the gap."),
    ("human", "Skills to assess: {skills_to_assess}\n\n"
              "Candidate answers:\n{answers}\n\n"
              "{format_instructions}"),
]).partial(format_instructions=ex3_parser2.get_format_instructions())

ex3_step2_chain = ex3_step2_prompt | llm | ex3_parser2

# ----- Step 3: final scorecard -----
class ScorecardRow(BaseModel):
    skill: str = Field(description="Skill name")
    rating: int = Field(description="Score from 1 (very weak) to 5 (excellent)")
    demonstrated: bool = Field(description="Whether the skill was clearly demonstrated")
    comments: str = Field(description="One short line of comment for the panel")

class Scorecard(BaseModel):
    rows: List[ScorecardRow] = Field(description="One row for each skill")
    overall_recommendation: str = Field(description="One of: Strong Hire, Hire, Borderline, No Hire - with a short reason")

ex3_parser3 = PydanticOutputParser(pydantic_object=Scorecard)

ex3_step3_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a hiring panel coordinator. You turn skill evaluations into a clear hiring scorecard "
               "and give a fair overall recommendation. Skills that were not demonstrated must get a low rating."),
    ("human", "Skill evaluation:\n{evaluation}\n\n{format_instructions}"),
]).partial(format_instructions=ex3_parser3.get_format_instructions())

ex3_step3_chain = ex3_step3_prompt | llm | ex3_parser3


def run_exercise_3(transcript_text, skills_to_assess):
    answers = ex3_step1_chain.invoke({"transcript_text": transcript_text})
    evaluation = ex3_step2_chain.invoke({
        "skills_to_assess": ", ".join(skills_to_assess),
        "answers": answers.model_dump_json(indent=2),
    })
    scorecard = ex3_step3_chain.invoke({"evaluation": evaluation.model_dump_json(indent=2)})
    return answers, evaluation, scorecard

In [ ]:
transcript_text = """Interviewer: Tell me about a project where you used Python.
Candidate: I made a student attendance system in Python with Flask and SQLite. I wrote the login, the
attendance marking pages and a report that exports to Excel.
Interviewer: How do you handle a bug that you cannot find?
Candidate: I add print statements and read the error carefully. Then I search for the error online and
if it still doesn't work I ask a senior or a friend.
Interviewer: Can you explain what a REST API is?
Candidate: Umm, I have heard about it but I haven't really worked with it. I think it is something to
connect frontend and backend.
Interviewer: How do you work in a team?
Candidate: In my final year project we were 4 members. I managed the database part and we used GitHub
to share code and I helped others when they were stuck."""

skills_to_assess = ["Python programming", "Debugging", "REST APIs", "Teamwork", "SQL / Databases"]

answers, evaluation, scorecard = run_exercise_3(transcript_text, skills_to_assess)

print("STEP 1 - Extracted answers")
for pair in answers.qa_pairs:
    print("Q:", pair.question)
    print("A:", pair.answer, "\n")

print("STEP 2 - Skill evaluation")
for item in evaluation.evaluations:
    flag = "OK  " if item.demonstrated else "FLAG"
    print(f"[{flag}] {item.skill}: {item.evidence_or_gap}")

print("\nSTEP 3 - Final scorecard")
for row in scorecard.rows:
    print(f"{row.skill:<20} rating={row.rating}  demonstrated={row.demonstrated}  | {row.comments}")
print("\nOverall recommendation:", scorecard.overall_recommendation)

---
## Exercise 4 — Outdoor Event Plan to Weather-Adjusted Recommendation Pipeline

**Question:** Build a 3-step pipeline. Step 1 extracts event details (event type, planned date, location, indoor/outdoor preference) from a free-form event description. Step 2 calls a weather API for the extracted location and date and normalizes the response into structured weather data. Step 3 generates a recommendation message for the organizer (proceed as planned / reschedule / move indoors) using both the event details and the weather data.

**Input:** `event_description_text`
**Output:** structured event details (Step 1), structured weather data (Step 2), recommendation message (Step 3)

**My approach**
- Step 1: `PromptTemplate` + `PydanticOutputParser`. Today's date is given to the prompt so words like "next Saturday" can be converted to a `YYYY-MM-DD` date.
- Step 2: A `PromptTemplate` formats the weather query from the Step 1 fields. The data comes from the real **Open-Meteo** API (free, no API key needed), not from the LLM. The API response is converted into JSON and parsed by `PydanticOutputParser` into `temperature`, `precipitation_chance`, `conditions`.
- Step 3: `ChatPromptTemplate` ("event planning assistant") + `StrOutputParser`.

**Note:** Open-Meteo forecast only works for dates within the next ~16 days, so the event date should be in that range. Internet access is needed for this exercise.

In [ ]:
# ----- Step 1: extract event details -----
class EventDetails(BaseModel):
    event_type: str = Field(description="Type of the event, e.g. wedding, picnic, cricket match")
    event_date: str = Field(description="Planned date in YYYY-MM-DD format")
    location: str = Field(description="City where the event will be held")
    indoor_outdoor_preference: str = Field(description="indoor, outdoor or flexible")

ex4_parser1 = PydanticOutputParser(pydantic_object=EventDetails)

ex4_step1_prompt = PromptTemplate(
    template=(
        "Extract the event details from the description below.\n"
        "Today's date is {today}. Convert any relative date (like 'next Saturday') into YYYY-MM-DD.\n"
        "For location give only the city name.\n"
        "{format_instructions}\n\n"
        "Description:\n{event_description_text}"
    ),
    input_variables=["event_description_text"],
    partial_variables={
        "today": date.today().isoformat(),
        "format_instructions": ex4_parser1.get_format_instructions(),
    },
)
ex4_step1_chain = ex4_step1_prompt | llm | ex4_parser1

In [ ]:
# ----- Step 2: weather API call + normalisation -----
class WeatherData(BaseModel):
    temperature: float = Field(description="Maximum temperature in degrees Celsius")
    precipitation_chance: int = Field(description="Chance of rain in percent (0-100)")
    conditions: str = Field(description="Short weather description, e.g. 'Light rain'")

ex4_parser2 = PydanticOutputParser(pydantic_object=WeatherData)

# This template only formats the weather query (the LLM is not used in this step)
ex4_query_prompt = PromptTemplate.from_template(
    "Daily forecast for city={location} on date={event_date}"
)

# Open-Meteo weather codes -> readable text (only the common ones)
WEATHER_CODES = {
    0: "Clear sky", 1: "Mainly clear", 2: "Partly cloudy", 3: "Overcast",
    45: "Fog", 48: "Fog", 51: "Light drizzle", 53: "Drizzle", 55: "Heavy drizzle",
    61: "Light rain", 63: "Rain", 65: "Heavy rain", 80: "Light rain showers",
    81: "Rain showers", 82: "Violent rain showers", 95: "Thunderstorm",
    96: "Thunderstorm with hail", 99: "Thunderstorm with hail",
}

def get_weather(event: EventDetails) -> WeatherData:
    query = ex4_query_prompt.format(location=event.location, event_date=event.event_date)
    print("Weather query ->", query)

    # 1) city name -> latitude/longitude
    geo = requests.get(
        "https://geocoding-api.open-meteo.com/v1/search",
        params={"name": event.location.split(",")[0].strip(), "count": 1},
        timeout=15,
    ).json()
    if "results" not in geo:
        raise ValueError(f"Could not find the location: {event.location}")
    place = geo["results"][0]

    # 2) forecast for that day
    forecast = requests.get(
        "https://api.open-meteo.com/v1/forecast",
        params={
            "latitude": place["latitude"],
            "longitude": place["longitude"],
            "daily": "temperature_2m_max,precipitation_probability_max,weathercode",
            "start_date": event.event_date,
            "end_date": event.event_date,
            "timezone": "auto",
        },
        timeout=15,
    ).json()
    if "daily" not in forecast:
        raise ValueError("No forecast available for this date (it must be within the next ~16 days). "
                         + str(forecast.get("reason", "")))

    daily = forecast["daily"]
    raw_weather = {
        "temperature": daily["temperature_2m_max"][0],
        "precipitation_chance": daily["precipitation_probability_max"][0] or 0,
        "conditions": WEATHER_CODES.get(daily["weathercode"][0], "Unknown"),
    }
    # normalise the API data using the output parser
    return ex4_parser2.parse(json.dumps(raw_weather))

In [ ]:
# ----- Step 3: recommendation message -----
ex4_step3_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are an event planning assistant. You advise organizers using the event details and the "
               "weather forecast. Always choose exactly one option: proceed as planned, reschedule, or move indoors."),
    ("human", "Event details:\n{event_details}\n\nWeather forecast:\n{weather_data}\n\n"
              "Write a short recommendation message for the organizer with the decision and the reason."),
])
ex4_step3_chain = ex4_step3_prompt | llm | StrOutputParser()


def run_exercise_4(event_description_text):
    event = ex4_step1_chain.invoke({"event_description_text": event_description_text})
    weather = get_weather(event)
    message = ex4_step3_chain.invoke({
        "event_details": event.model_dump_json(indent=2),
        "weather_data": weather.model_dump_json(indent=2),
    })
    return event, weather, message

In [ ]:
event_description_text = """We are planning a college farewell party in Pune on the coming Saturday evening.
We want to do it on the open lawn behind the department building, around 150 students will come."""

event, weather, message = run_exercise_4(event_description_text)

print("\nSTEP 1 - Event details")
print(event.model_dump_json(indent=2))
print("\nSTEP 2 - Weather data")
print(weather.model_dump_json(indent=2))
print("\nSTEP 3 - Recommendation")
print(message)

---
## Exercise 5 — Customer Support Chat Log to Escalation Report Pipeline

**Question:** Build a 3-step pipeline. Step 1 normalizes a messy support chat log into a structured summary (issue, customer tone, resolution attempted). Step 2 checks the structured summary for missing information needed to escalate (e.g. account ID, error code) and flags the gaps. Step 3 generates a prioritized escalation report for the support lead based on the structured, gap-checked version.

**Input:** `chat_log_text`
**Output:** structured summary (Step 1), flagged gaps (Step 2), escalation report (Step 3)

**My approach**
- Step 1: `PromptTemplate` + `PydanticOutputParser` (`issue`, `customer_tone`, `resolution_attempted`). Since the schema has no separate field for account ID or error code, I ask the model to write them inside the `issue` field if they are present in the chat.
- Step 2: `ChatPromptTemplate` ("escalation-readiness reviewer") + `PydanticOutputParser` (`missing_fields`, `is_ready_to_escalate`).
- Step 3: `ChatPromptTemplate` ("support operations lead") + `StrOutputParser`.

In [ ]:
# ----- Step 1: normalise chat log -----
class ChatSummary(BaseModel):
    issue: str = Field(description="The customer's problem. Include account ID and error code here if they were mentioned")
    customer_tone: str = Field(description="Tone of the customer, e.g. calm, frustrated, angry")
    resolution_attempted: str = Field(description="What the support agent already tried")

ex5_parser1 = PydanticOutputParser(pydantic_object=ChatSummary)

ex5_step1_prompt = PromptTemplate(
    template=(
        "The chat log below is messy. Convert it into a clean structured summary.\n"
        "If the customer or agent mentions an account ID, order ID or error code, write it inside the issue field.\n"
        "{format_instructions}\n\n"
        "Chat log:\n{chat_log_text}"
    ),
    input_variables=["chat_log_text"],
    partial_variables={"format_instructions": ex5_parser1.get_format_instructions()},
)
ex5_step1_chain = ex5_step1_prompt | llm | ex5_parser1

# ----- Step 2: check for missing information -----
class GapCheck(BaseModel):
    missing_fields: List[str] = Field(description="Information needed for escalation that is missing, e.g. account ID, error code")
    is_ready_to_escalate: bool = Field(description="True only if nothing important is missing")

ex5_parser2 = PydanticOutputParser(pydantic_object=GapCheck)

ex5_step2_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are an escalation-readiness reviewer. To escalate an issue the team needs at least: "
               "account ID, error code or error message, and the steps already tried. "
               "Check the summary and list whatever is missing."),
    ("human", "Structured summary:\n{summary}\n\n{format_instructions}"),
]).partial(format_instructions=ex5_parser2.get_format_instructions())

ex5_step2_chain = ex5_step2_prompt | llm | ex5_parser2

# ----- Step 3: escalation report -----
ex5_step3_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a support operations lead. You write prioritized escalation reports for the support lead."),
    ("human", "Structured summary:\n{summary}\n\nGap check:\n{gap_check}\n\n"
              "Write an escalation report with: priority (High/Medium/Low) with a reason, issue overview, "
              "customer tone, what was already tried, missing information to collect, and the next action."),
])
ex5_step3_chain = ex5_step3_prompt | llm | StrOutputParser()


def run_exercise_5(chat_log_text):
    summary = ex5_step1_chain.invoke({"chat_log_text": chat_log_text})
    gaps = ex5_step2_chain.invoke({"summary": summary.model_dump_json(indent=2)})
    report = ex5_step3_chain.invoke({
        "summary": summary.model_dump_json(indent=2),
        "gap_check": gaps.model_dump_json(indent=2),
    })
    return summary, gaps, report

In [ ]:
chat_log_text = """[10:02] cust: hi my payment failed 3 times today!!! money got deducted tho
[10:03] agent: Sorry to hear that. Can you tell me what error you saw?
[10:04] cust: it just says something went wrong. this is the 3rd time i'm facing this, very annoying
[10:06] agent: Please try clearing cache and use another browser
[10:09] cust: did that already, same thing. i want my money back or the payment to go through
[10:10] agent: I will check with my team and get back."""

summary, gaps, report = run_exercise_5(chat_log_text)

print("STEP 1 - Structured summary")
print(summary.model_dump_json(indent=2))
print("\nSTEP 2 - Gap check")
print(gaps.model_dump_json(indent=2))
print("\nSTEP 3 - Escalation report")
print(report)

---
## Exercise 6 — Workout Goal to Weekly Plan Pipeline

**Question:** Build a 2-step pipeline. Step 1 extracts fitness goal details (goal type, current fitness level, days available per week, equipment access) from a free-form fitness goal description. Step 2 generates a structured weekly workout plan scaled to the extracted availability, using only the extracted details from Step 1, not the original description.

**Input:** `fitness_goal_text`, `days_per_week`
**Output:** extracted goal details (Step 1) and weekly workout plan (Step 2)

**My approach**
- Step 1: `PromptTemplate` + `PydanticOutputParser` (`goal_type`, `fitness_level`, `days_available`, `equipment_access`). The `days_per_week` input is given to the prompt too, so `days_available` follows it.
- Step 2: `ChatPromptTemplate` ("certified personal trainer") + `PydanticOutputParser` (list of day-by-day plan entries).

In [ ]:
# ----- Step 1: extract fitness goal details -----
class FitnessGoal(BaseModel):
    goal_type: str = Field(description="Goal, e.g. weight loss, muscle gain, stamina")
    fitness_level: str = Field(description="beginner, intermediate or advanced")
    days_available: int = Field(description="Number of days per week the person can workout (1 to 7)")
    equipment_access: str = Field(description="Equipment available, e.g. none, dumbbells, full gym")

ex6_parser1 = PydanticOutputParser(pydantic_object=FitnessGoal)

ex6_step1_prompt = PromptTemplate(
    template=(
        "Extract the fitness goal details from the text below.\n"
        "The person says they can workout {days_per_week} days per week, so use this number for days_available.\n"
        "{format_instructions}\n\n"
        "Fitness goal text:\n{fitness_goal_text}"
    ),
    input_variables=["fitness_goal_text", "days_per_week"],
    partial_variables={"format_instructions": ex6_parser1.get_format_instructions()},
)
ex6_step1_chain = ex6_step1_prompt | llm | ex6_parser1

# ----- Step 2: weekly plan (uses only Step 1 data) -----
class DayPlan(BaseModel):
    day: str = Field(description="Day of the week, e.g. Monday")
    focus: str = Field(description="Focus of the day, e.g. Upper body, Cardio, Rest")
    exercises: List[str] = Field(description="Exercises with sets x reps or duration. Empty list for a rest day")
    duration_minutes: int = Field(description="Total time of the session in minutes (0 for a rest day)")

class WeeklyPlan(BaseModel):
    plan: List[DayPlan] = Field(description="Plan for all 7 days of the week")

ex6_parser2 = PydanticOutputParser(pydantic_object=WeeklyPlan)

ex6_step2_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a certified personal trainer. You make safe and realistic weekly workout plans "
               "that match the person's level, equipment and available days."),
    ("human", "Goal details:\n{goal_details}\n\n"
              "Make a weekly plan for all 7 days. Use exactly days_available workout days and mark the remaining days as rest. "
              "Only use the equipment mentioned.\n\n{format_instructions}"),
]).partial(format_instructions=ex6_parser2.get_format_instructions())

ex6_step2_chain = ex6_step2_prompt | llm | ex6_parser2


def run_exercise_6(fitness_goal_text, days_per_week):
    goal = ex6_step1_chain.invoke({"fitness_goal_text": fitness_goal_text, "days_per_week": days_per_week})
    weekly_plan = ex6_step2_chain.invoke({"goal_details": goal.model_dump_json(indent=2)})
    return goal, weekly_plan

In [ ]:
fitness_goal_text = """I want to lose around 6 kg in the next three months. I'm a beginner, I never went to a gym before.
I only have a pair of 5 kg dumbbells and a yoga mat at home."""
days_per_week = 4

goal, weekly_plan = run_exercise_6(fitness_goal_text, days_per_week)

print("STEP 1 - Extracted goal details")
print(goal.model_dump_json(indent=2))

print("\nSTEP 2 - Weekly workout plan")
for d in weekly_plan.plan:
    print(f"\n{d.day} - {d.focus} ({d.duration_minutes} min)")
    for ex in d.exercises:
        print("   -", ex)

---
## Result / Conclusion
In this lab I built six LLM workflows using `PromptTemplate`, `ChatPromptTemplate`, `StrOutputParser` and `PydanticOutputParser`. Extracting data into Pydantic classes made each step's output clean and typed, so it could be passed directly into the next prompt without any manual cleanup. Choosing the persona (system message) for the generation and evaluation steps made the replies more focused, and in Exercise 4 the weather data came from a real API instead of the LLM, which avoids made-up values.

## Pushing to GitHub
Run these in a terminal inside the repository folder after finishing each exercise (replace the names):
```bash
git add Lab5_Workflow_Design_with_LLMs.ipynb
git commit -m "Lab 5: workflow design with LLMs"
git push origin main
```
Repository link: `https://github.com/AbhishekKambleGit/GenAI_llm_engineering_scfp125072`